# Continuously Tempered Diffusion Sampler
In this notebook, we train continuously tempered diffusion samplers (CTDS) for linear and interpolant annealing paths between a Gaussian an a 40-mode Gaussian mixture target [1]. 

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../')

import torch
from omegaconf import OmegaConf
from matplotlib import pyplot as plt
import seaborn as sns

from src.models.ctds import CTDSModule
from src.utils.train import train_module
from src.utils.plotting import mt_plot_proposal, mt_timeseries_plot_density, mt_timeseries_plot_contours

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
def plotter_helper(ctds_module: CTDSModule, proposal_name: str):
    ctds_module.move_to_device(device)
    proposal = ctds_module.build_mt_proposal(proposal_name).to(device)
    proposal.record_every = 125
    t_bins = torch.linspace(0, 1, 5).to(device)
    b_bins = torch.linspace(0.2, 1.0, 3).to(device)
    scale = 50.0
    sample_results, axes = mt_plot_proposal(
        proposal = proposal,
        num_samples = 500,
        nts = 500,
        t_bins = t_bins,
        b_bins = b_bins,
        scale = scale
    )

    # Title the axes and set ticks to empty
    for i in range(len(b_bins)):
        for j in range(len(t_bins)):
            axes[i, j].set_xticks([])
            axes[i, j].set_yticks([])
            axes[i, j].set_title(axes[i, j].get_title(), fontsize=20)

    # Set border ticks on temp axis
    for i in range(len(b_bins)):
        axes[i, 0].set_yticks([-40, -20, 0, 20, 40])
        axes[i, 0].set_yticklabels([-40, -20, 0, 20, 40], fontsize=15)
        axes[i, 0].set_ylabel(r'$\beta=$' + f'{b_bins[i].item():.2f}', fontsize=25)

    # Set border ticks on time axis
    for i in range(len(t_bins)):
        axes[len(b_bins)-1, i].set_xticks([-40, -20, 0, 20, 40])
        axes[len(b_bins)-1, i].set_xticklabels([-40, -20, 0, 20, 40], fontsize=15)
        axes[len(b_bins)-1, i].set_xlabel(f't={t_bins[i].item():.2f}', fontsize=25)

    mt_timeseries_plot_density(ctds_module.mt_continuum.log_density, ts=t_bins, bs=b_bins, scale=scale, axes=axes, alpha=0.5, zorder=0, vmin=-40, device=device)
    mt_timeseries_plot_contours(ctds_module.mt_continuum.log_density, ts=t_bins, bs=b_bins, scale=scale, axes=axes, alpha=0.5, legend=False, device=device)

def get_bs(ctds_module: CTDSModule, nts: int = 500, num_samples: int = 2500):
    """
    Sample from the CTDS dynamics.
    """
    dynamics = ctds_module.proposal.dynamics
    converter = ctds_module.proposal.converter
    ts = torch.linspace(0, 1, nts).to(device).view(1, -1, 1).expand(num_samples, -1, -1)
    xz_pxz, _ = dynamics.sample(
        ts=ts, num_samples=num_samples, use_tqdm = True
    )
    z = xz_pxz[:, 2]
    b = converter.xi_to_beta(z)
    return b

### CTDS on Interpolant Path

In [3]:
# Initialize NETS module from config
ctds_interpolant_cfg = OmegaConf.create(
    {
        # Run details
        "run_name": "ctds_interpolant_notebook_test",
        "run_group": "notebook_test",
        "wandb": False,
        "checkpoint": None,

        # Objective details
        "x_dim": 2,
        "mt_continuum": "from_density_path",
        "density_path": "interpolant",
        "target": "fab_gmm",
        "cov_scale": 1.0,
        "source_std": 2.0,

        # Training details
        "num_devices": 1,
        "max_epochs": 25,
        "lr": 0.002,
        "lr_burn_in_epochs": 0,
        "step_size": 1,
        "gamma": 0.8,
        "checkpoint_burn_in_epochs": 0,

        # Batching details
        "use_persistent_sample_buffer": True,
        "persistent_sample_buffer_trajectories": 5000,
        "train_batch_size": 10000,
        "train_steps_per_epoch": 500,
        "val_trajectories": 1000,
        "val_steps_per_epoch": 2,
        "val_freq": 1,

        # PINN-specific training details
        "annealing_scheduler": "constant",
        "start_T": 1.0,
        "avg_dt": 0.002,
        "record_every": 5,
        "divergence_mode": "autograd",

        # Control-specific details
        "x_control": "mlp",
        "x_control_hiddens": [256, 256, 256],

        # Free-energy specific details
        "free_energy": "mlp",
        "free_energy_hiddens": [256, 256, 256],

        # Time-encoding details
        "use_fourier": True,
        "x_fourier_dim": 100,
        "x_fourier_sigma": 0.1,
        "b_fourier_dim": 20,
        "b_fourier_sigma": 1,
        "t_fourier_dim": 20,
        "t_fourier_sigma": 5,

        # Proposal details
        "proposal": "underdamped_ctds",
        "x_scale": 50.0,
        "z_scale": 5.0,
        "x_damping": 2.0,
        "z_damping": 2.0,
        "hamiltonian_type": "tempered",
        "x_mass": 1.0,
        "z_mass": 1.0,
        "beta_min": 0.2,
        "beta_max": 1.0,
        "use_jarzynski": False,

        # Reparameterization details
        "converter": "polynomial",
        "z_lower_bound": -2.0,
        "z_upper_bound": 2.0,
        "delta": 0.25,
        "delta_prime": 1.9,
        "s": 0.8,
        "enforce_z_bounds": False,

        # Confining potential details
        "confining_sharpness": 10.0,
        "confining_lower_bound": -2.0,
        "confining_upper_bound": 2.0,

        # Biasing details
        "biasing_density": "zero",

        # Debug
        "verbose": False,
    }
)

ctds_interpolant_module = CTDSModule(cfg=ctds_interpolant_cfg).move_to_device(device)

In [ ]:
# Untrained transport
plotter_helper(ctds_interpolant_module, "ode")

In [ ]:
# Untrained proposal (transport + tempered Langevin)
plotter_helper(ctds_interpolant_module, "underdamped_ctds")

In [ ]:
# Histogram of inverse-temperatures (betas) from untrained proposal
plt.figure(figsize=(8, 4), dpi=100)
ctds_interpolant_module.move_to_device(device)
untrained_bs = get_bs(ctds_interpolant_module, nts=500, num_samples=2500).cpu().numpy()
sns.histplot(untrained_bs, color="orange", label="Untrained CTDS Proposal", kde=False, stat="density", binrange=(0.2, 1.0), bins=20, alpha=0.5)

plt.xlim(0.2, 1.0)
plt.xlabel("Inverse Temperature", fontsize=12)
plt.ylabel("Density", fontsize=12)
plt.legend(prop={'size': 12})
plt.title("Inverse Temperature of Inertial CTDS Proposal Samples at T=1.0")
plt.show()

In [ ]:
# Now, let's train the CTDS module!
train_module(
    module=ctds_interpolant_module,
    run_name=ctds_interpolant_cfg.run_name,
    run_group=ctds_interpolant_cfg.run_group,
    max_epochs=ctds_interpolant_cfg.max_epochs,
    val_freq=ctds_interpolant_cfg.val_freq,
    wandb=False,
)

In [ ]:
# Trained transport
plotter_helper(ctds_interpolant_module, "ode")

In [ ]:
# Trained proposal (transport + tempered Langevin)
plotter_helper(ctds_interpolant_module, "underdamped_ctds")

In [ ]:
# Histogram of inverse-temperatures (betas) from trained proposal
plt.figure(figsize=(8, 4), dpi=100)
ctds_interpolant_module.move_to_device(device)
trained_bs = get_bs(ctds_interpolant_module, nts=500, num_samples=2500).cpu().numpy()
sns.histplot(trained_bs, color="orange", label="Trained CTDS Proposal", kde=False, stat="density", binrange=(0.2, 1.0), bins=20, alpha=0.5)

plt.xlim(0.2, 1.0)
plt.xlabel("Inverse Temperature", fontsize=12)
plt.ylabel("Density", fontsize=12)
plt.legend(prop={'size': 12})
plt.title("Inverse Temperature of Inertial CTDS Proposal Samples at T=1.0")
plt.show()

### CTDS on Linear Path
We can now conduct the same training process on the more challenging linear path. **Warning**: Doing so takes at least an order of magnitude more time!

In [ ]:
# Initialize NETS module from config
ctds_linear_cfg = OmegaConf.create(
    {
        # Run details
        "run_name": "ctds_linear_notebook_test",
        "run_group": "notebook_test",
        "wandb": False,
        "checkpoint": None,

        # Objective details
        "x_dim": 2,
        "mt_continuum": "learnable_linear",
        "learnable_hiddens": [256, 256, 256],
        "target": "fab_gmm",
        "cov_scale": 1.0,
        "source_std": 5.0,

        # Training details
        "num_devices": 1,
        "max_epochs": 800,
        "lr": 0.002,
        "lr_burn_in_epochs": 150,
        "step_size": 5,
        "gamma": 0.97,
        "checkpoint_burn_in_epochs": 150,

        # Batching details
        "use_persistent_sample_buffer": True,
        "persistent_sample_buffer_trajectories": 2000,
        "train_batch_size": 6250,
        "train_steps_per_epoch": 100,
        "val_trajectories": 1000,
        "val_steps_per_epoch": 2,
        "val_freq": 5,

        # PINN-specific training details
        "annealing_scheduler": "manual",
        "T_schedule": [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
        "epochs_per_T": [10, 10, 10, 10, 20, 20, 20, 30, 30, 30],
        "avg_dt": 0.002,
        "record_every": 20,
        "divergence_mode": "autograd",

        # Control-specific details
        "x_control": "mlp",
        "x_control_hiddens": [256, 256, 256],

        # Free-energy specific details
        "free_energy": "mlp",
        "free_energy_hiddens": [256, 256, 256],

        # Time-encoding details
        "use_fourier": True,
        "x_fourier_dim": 100,
        "x_fourier_sigma": 0.1,
        "b_fourier_dim": 20,
        "b_fourier_sigma": 1,
        "t_fourier_dim": 20,
        "t_fourier_sigma": 5,

        # Proposal details
        "proposal": "underdamped_ctds",
        "x_scale": 50.0,
        "z_scale": 5.0,
        "x_damping": 2.0,
        "z_damping": 2.0,
        "hamiltonian_type": "tempered",
        "x_mass": 1.0,
        "z_mass": 1.0,
        "beta_min": 0.2,
        "beta_max": 1.0,
        "use_jarzynski": True,

        # Reparameterization details
        "converter": "polynomial",
        "z_lower_bound": -2.0,
        "z_upper_bound": 2.0,
        "delta": 0.25,
        "delta_prime": 1.9,
        "s": 0.8,
        "enforce_z_bounds": False,

        # Confining potential details
        "confining_sharpness": 10.0,
        "confining_lower_bound": -2.0,
        "confining_upper_bound": 2.0,

        # Biasing details
        "biasing_density": "zero",

        # Debug
        "verbose": False,
    }
)

ctds_linear_module = CTDSModule(cfg=ctds_linear_cfg).move_to_device(device)

In [ ]:
# Untrained transport
plotter_helper(ctds_linear_module, "ode")

In [ ]:
# Untrained proposal (transport + tempered Langevin)
plotter_helper(ctds_linear_module, "underdamped_ctds")

In [ ]:
# Histogram of inverse-temperatures (betas) from untrained proposal
plt.figure(figsize=(8, 4), dpi=100)
ctds_linear_module.move_to_device(device)
untrained_bs = get_bs(ctds_linear_module, nts=500, num_samples=2500).cpu().numpy()
sns.histplot(untrained_bs, color="orange", label="Untrained CTDS Proposal", kde=False, stat="density", binrange=(0.2, 1.0), bins=20, alpha=0.5)

plt.xlim(0.2, 1.0)
plt.xlabel("Inverse Temperature", fontsize=12)
plt.ylabel("Density", fontsize=12)
plt.legend(prop={'size': 12})
plt.title("Inverse Temperature of Inertial CTDS Proposal Samples at T=1.0")
plt.show()

In [ ]:
# Now, let's train the CTDS module!
train_module(
    module=ctds_linear_module,
    run_name=ctds_linear_cfg.run_name,
    run_group=ctds_linear_cfg.run_group,
    max_epochs=ctds_linear_cfg.max_epochs,
    val_freq=5,
    wandb=False,
)

In [ ]:
# Trained transport
plotter_helper(ctds_linear_module, "ode")

In [ ]:
# Trained proposal (transport + tempered Langevin)
plotter_helper(ctds_linear_module, "underdamped_ctds")

In [ ]:
# Histogram of inverse-temperatures (betas) from trained proposal
plt.figure(figsize=(8, 4), dpi=100)
ctds_linear_module.move_to_device(device)
trained_bs = get_bs(ctds_linear_module, nts=500, num_samples=2500).cpu().numpy()
sns.histplot(trained_bs, color="orange", label="Trained CTDS Proposal", kde=False, stat="density", binrange=(0.2, 1.0), bins=20, alpha=0.5)

plt.xlim(0.2, 1.0)
plt.xlabel("Inverse Temperature", fontsize=12)
plt.ylabel("Density", fontsize=12)
plt.legend(prop={'size': 12})
plt.title("Inverse Temperature of Inertial CTDS Proposal Samples at T=1.0")
plt.show()

### References
1. Laurence Illing Midgley, Vincent Stimper, Gregor N. C. Simm, Bernhard Schölkopf, and José Miguel
Hernández-Lobato. Flow annealed importance sampling bootstrap. In The Eleventh International
Conference on Learning Representations, 2023. URL https://openreview.net/forum?id=XCTVFJwS9LJ.